# Notebook 15 — Experiment 23: Misclassification Error Taxonomy
### Novelty 3
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Experiment 17 showed *where* errors concentrate. This one explains *why*.

Fifty confident misclassifications are each assigned to one of four named
categories using combined SHAP and Anchor evidence:

| Type | Name | Signature |
|---|---|---|
| 1 | Lexical Trap | Top SHAP feature is a strongly sentimental word; Anchor confirms that word alone forces the prediction |
| 2 | Emoji Polarity Confusion | Emoji density dominates SHAP and the emoji's usual polarity contradicts the true label |
| 3 | Sarcasm Inversion | True negative, predicted positive; Anchor fires on positive surface vocabulary with no negation present |
| 4 | Noise and Slang | Anchor precision below 0.60 — no reliable rule exists; SHAP is dominated by punctuation or slang counts |

The decisive question is whether one type dominates **across model families**. If
it does, the cause sits in the TF-IDF representation and no amount of model
selection will fix it. If the distribution varies by model, targeted model choice
can help. Those two conclusions lead to completely different recommendations.

Also computes the Explanation Agreement Score — Jaccard similarity between top
SHAP features across model pairs — which is the direct evidence for that claim.

Zero training runs. Fills **Table 12**.

## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
!pip install -q shap lime anchor-exp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.6 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.3/427.3 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Cell 2: Load Misclassified Instances

Notebook 8 saved these with their original text attached. If that file is
missing, run Notebook 8 first.

In [2]:
D = PATHS["data"]

mis = pd.read_parquet(D / "misclassified_instances.parquet")

with open(D / "final_model_config.json") as f:
    cfg = json.load(f)

final_model = load_model("exp10", "FinalModel")
with open(D / "final_transformers.pkl", "rb") as f:
    transformers = pickle.load(f)
vec = transformers[0] if isinstance(transformers, tuple) else transformers

print(f"Misclassified instances : {len(mis):,}")
print(f"Model                   : {cfg['model']}\n")
print("Errors by subgroup:")
print(mis["subgroup"].value_counts().to_string())

Misclassified instances : 4,937
Model                   : LogisticRegression

Errors by subgroup:
subgroup
formal         4242
other           405
emoji-heavy     264
slang-heavy      23
sarcasm           3


## Cell 3: Stratified Sample

Fifty instances, with sarcasm guaranteed at least fifteen. Without the
stratification the sample would be almost entirely formal posts, since those are
95% of the data.

In [3]:
QUOTA = {"sarcasm": 15, "emoji-heavy": 10, "slang-heavy": 10, "formal": 10, "mixed": 5}

parts = []
for sg, n in QUOTA.items():
    avail = mis[mis["subgroup"] == sg].nlargest(n, "confidence")
    if len(avail):
        parts.append(avail)
        print(f"  {sg:<14}: {len(avail):>2} of {n} requested")

sample = pd.concat(parts, ignore_index=True)
print(f"\nTotal sample: {len(sample)} instances")
print(f"Mean confidence: {sample['confidence'].mean():.3f}")

  sarcasm       :  3 of 15 requested
  emoji-heavy   : 10 of 10 requested
  slang-heavy   : 10 of 10 requested
  formal        : 10 of 10 requested

Total sample: 33 instances
Mean confidence: 0.905


## Cell 4: SHAP Per Instance

Feature attributions for each sampled error. These drive the Type 1 and Type 2
classification rules.

In [6]:
print("Columns currently in sample:")
print(sorted(sample.columns.tolist()))

SOCIAL_COLS = ["emoji_density", "slang_ratio", "punct_intensity"]

missing = [c for c in SOCIAL_COLS if c not in sample.columns]
if missing:
    print(f"\nMissing {missing} — recomputing from text.")
    sample = tag_subgroups(sample, text_col="text")
    print("Recomputed.")
else:
    print("\nAll linguistic features present.")

print(f"\nsample shape: {sample.shape}")
print(sample[SOCIAL_COLS].describe().round(4).to_string())

Columns currently in sample:
['confidence', 'correct', 'proba_0', 'proba_1', 'proba_2', 'shap_top', 'subgroup', 'text', 'text_clean', 'top_shap_feature', 'y_pred', 'y_true']

Missing ['emoji_density', 'slang_ratio', 'punct_intensity'] — recomputing from text.
Recomputed.

sample shape: (33, 28)
       emoji_density  slang_ratio  punct_intensity
count        33.0000      33.0000          33.0000
mean          0.0493       0.0483           0.0409
std           0.0914       0.0748           0.0888
min           0.0000       0.0000           0.0000
25%           0.0000       0.0000           0.0000
50%           0.0000       0.0000           0.0000
75%           0.0625       0.1111           0.0556
max           0.3333       0.2500           0.4000


In [7]:
import numpy as np
import scipy.sparse as sp

SOCIAL_COLS = ["emoji_density", "slang_ratio", "punct_intensity"]

# ── rebuild the feature space the model was actually trained on ────
if isinstance(transformers, tuple) and len(transformers) == 3:
    vec_w, vec_c, scaler = transformers
    Xw = vec_w.transform(sample["text_clean"])
    Xc = vec_c.transform(sample["text_clean"])
    Xn = scaler.transform(sample[SOCIAL_COLS].values)
    X_sample = sp.hstack([Xw, Xc, sp.csr_matrix(Xn)]).tocsr()
    feature_names = np.concatenate([
        vec_w.get_feature_names_out(),
        np.array([f"char_{f}" for f in vec_c.get_feature_names_out()]),
        np.array(SOCIAL_COLS),
    ])
elif isinstance(transformers, tuple) and len(transformers) == 2:
    vec_w, scaler = transformers
    Xw = vec_w.transform(sample["text_clean"])
    Xn = scaler.transform(sample[SOCIAL_COLS].values)
    X_sample = sp.hstack([Xw, sp.csr_matrix(Xn)]).tocsr()
    feature_names = np.concatenate([
        vec_w.get_feature_names_out(), np.array(SOCIAL_COLS)])
else:
    X_sample = transformers.transform(sample["text_clean"])
    feature_names = np.array(transformers.get_feature_names_out())

print(f"Feature matrix : {X_sample.shape}")

# ── coefficients ───────────────────────────────────────────────────
inner = final_model
if hasattr(final_model, "calibrated_classifiers_"):
    inner = final_model.calibrated_classifiers_[0].estimator

if hasattr(inner, "coef_"):
    coefs = np.asarray(inner.coef_)
elif hasattr(inner, "feature_log_prob_"):
    coefs = np.asarray(inner.feature_log_prob_)
else:
    coefs = None

print(f"Coefficients   : {coefs.shape if coefs is not None else 'tree model'}")

shap_top = []

if coefs is not None:
    assert coefs.shape[1] == X_sample.shape[1], (
        f"MISMATCH: model {coefs.shape[1]:,} vs matrix {X_sample.shape[1]:,}")

    # exact SHAP for a linear model: coef_j * (x_ij - E[x_j])
    X_dense  = np.asarray(X_sample.todense())
    baseline = X_dense.mean(axis=0)
    centred  = X_dense - baseline

    print("Computing per-instance SHAP values...")
    for i in range(X_dense.shape[0]):
        contrib = np.zeros(X_dense.shape[1])
        for c in range(coefs.shape[0]):
            contrib += np.abs(centred[i] * coefs[c])
        contrib /= coefs.shape[0]
        top_idx = np.argsort(contrib)[-5:][::-1]
        shap_top.append([feature_names[j] for j in top_idx if contrib[j] > 0])
    print(f"  done for {len(shap_top)} instances")

else:
    import shap
    try:
        expl = shap.TreeExplainer(final_model)
        vals = expl.shap_values(X_sample)
        arr = (np.mean([np.abs(v) for v in vals], axis=0)
               if isinstance(vals, list) else np.abs(vals))
        if arr.ndim == 3:
            arr = arr.mean(axis=2)
        for i in range(arr.shape[0]):
            top_idx = np.argsort(arr[i])[-5:][::-1]
            shap_top.append([feature_names[j] for j in top_idx if arr[i][j] > 0])
        print(f"  done for {len(shap_top)} instances")
    except Exception as e:
        print(f"  TreeExplainer failed: {type(e).__name__} — {e}")
        shap_top = [[] for _ in range(len(sample))]

sample["shap_top"] = shap_top
sample["top_shap_feature"] = [t[0] if t else "" for t in shap_top]

print("\nTop SHAP feature by subgroup:")
for sg in sample["subgroup"].unique():
    part = sample[sample["subgroup"] == sg]
    feats = [f for f in part["top_shap_feature"] if f][:5]
    print(f"  {sg:<14}: {', '.join(feats)}")

Feature matrix : (33, 80003)
Coefficients   : (3, 80003)
Computing per-instance SHAP values...
  done for 33 instances

Top SHAP feature by subgroup:
  sarcasm       : char_s? , for sure, not
  emoji-heavy   : emoji, do you, happy, die, is back
  slang-heavy   : punct_intensity, love, win, thank, the end
  formal        : looking forward, doctor, of love, exciting, great


## Cell 5: Anchor Rules

Anchor produces a minimal rule sufficient to force the prediction, with a
precision score. Precision below 0.60 means no reliable rule exists, which is the
signature of a Type 4 error.

In [8]:
from anchor import anchor_text
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm -q
    nlp = spacy.load("en_core_web_sm")

classes = sorted(sample["y_true"].unique())

def predict_fn(texts):
    return final_model.predict(vec.transform(texts))

anchor_explainer = anchor_text.AnchorText(nlp, classes, use_unk_distribution=True)

ANCHOR_N = 20     # most confident sarcasm errors
anchor_sample = sample[sample["subgroup"] == "sarcasm"].nlargest(ANCHOR_N, "confidence")

anchor_results = {}
print(f"Generating Anchor rules for {len(anchor_sample)} sarcasm errors...\n")

for idx, r in anchor_sample.iterrows():
    try:
        exp = anchor_explainer.explain_instance(
            str(r["text_clean"]), predict_fn, threshold=0.90,
            use_proba=False, batch_size=10, max_anchor_size=3)
        rule = " AND ".join(exp.names())
        anchor_results[idx] = {"rule": rule,
                               "precision": exp.precision(),
                               "coverage": exp.coverage()}
        print(f"  {r['y_true']} -> {r['y_pred']}: IF {rule}")
        print(f"      precision {exp.precision():.2f}  coverage {exp.coverage():.2f}")
    except Exception as e:
        anchor_results[idx] = {"rule": "", "precision": np.nan, "coverage": np.nan}

sample["anchor_rule"]      = sample.index.map(lambda i: anchor_results.get(i, {}).get("rule", ""))
sample["anchor_precision"] = sample.index.map(lambda i: anchor_results.get(i, {}).get("precision", np.nan))
sample["anchor_coverage"]  = sample.index.map(lambda i: anchor_results.get(i, {}).get("coverage", np.nan))

Generating Anchor rules for 3 sarcasm errors...



## Cell 6: Assign Error Types

Rules applied in order. Type 3 is checked first because sarcasm inversion is the
most specific pattern and the one this study exists to investigate.

In [9]:
SENTIMENT_WORDS = {
    "great","good","amazing","awesome","love","best","perfect","excellent",
    "wonderful","fantastic","brilliant","lovely","nice","happy","beautiful",
    "bad","terrible","awful","worst","hate","horrible","poor","disappointing",
}
NEGATIONS = {"not","no","never","cannot","cant","wont","dont","didnt","isnt","arent"}

def classify_error(r):
    top   = str(r["top_shap_feature"]).lower()
    rule  = str(r["anchor_rule"]).lower()
    prec  = r["anchor_precision"]
    text  = str(r["text_clean"]).lower()
    sg    = r["subgroup"]

    # Type 3 — sarcasm inversion
    if (r["y_true"] == "negative" and r["y_pred"] == "positive"
            and not any(n in text.split() for n in NEGATIONS)
            and any(w in text for w in SENTIMENT_WORDS)):
        return "Type 3", "Sarcasm Inversion"

    # Type 4 — no reliable rule found
    if (not np.isnan(prec) and prec < 0.60) or \
       any(k in top for k in ["punct", "slang", "!", "?"]):
        return "Type 4", "Noise and Slang Distortion"

    # Type 2 — emoji polarity
    if sg == "emoji-heavy" or "emoji" in top:
        return "Type 2", "Emoji Polarity Confusion"

    # Type 1 — lexical trap
    if any(w in top for w in SENTIMENT_WORDS) or \
       any(w in rule for w in SENTIMENT_WORDS):
        return "Type 1", "Lexical Trap"

    return "Type 1", "Lexical Trap"     # default

sample[["taxonomy_type", "taxonomy_name"]] = sample.apply(
    lambda r: pd.Series(classify_error(r)), axis=1)

print("ERROR TYPE DISTRIBUTION")
print("="*60)
print(sample["taxonomy_name"].value_counts().to_string())

print("\nBY SUBGROUP")
print("="*60)
print(pd.crosstab(sample["subgroup"], sample["taxonomy_name"]).to_string())

ERROR TYPE DISTRIBUTION
taxonomy_name
Lexical Trap                  18
Emoji Polarity Confusion      10
Noise and Slang Distortion     3
Sarcasm Inversion              2

BY SUBGROUP
taxonomy_name  Emoji Polarity Confusion  Lexical Trap  Noise and Slang Distortion  Sarcasm Inversion
subgroup                                                                                            
emoji-heavy                          10             0                           0                  0
formal                                0             9                           0                  1
sarcasm                               0             2                           1                  0
slang-heavy                           0             7                           2                  1


## Cell 7: Explanation Agreement Score

Jaccard similarity between the top-10 SHAP features of different model families.
High agreement means the models rely on the same signals, which is the evidence
that a shared failure mode belongs to the representation rather than to any one
algorithm.

In [10]:
try:
    profiles = np.load(PATHS["results"] / "shap_profiles.npy", allow_pickle=True).item()
    fnames   = np.load(PATHS["results"] / "shap_feature_names.npy", allow_pickle=True)

    def top_set(profile, k=10):
        return set(fnames[np.argsort(profile)[-k:]])

    subs = list(profiles.keys())
    rows = []
    for i, a in enumerate(subs):
        for b in subs[i+1:]:
            sa, sb = top_set(profiles[a]), top_set(profiles[b])
            jac = len(sa & sb) / len(sa | sb) if (sa | sb) else 0
            rows.append({"Subgroup A": a, "Subgroup B": b,
                         "Jaccard": round(jac, 3),
                         "Shared Features": ", ".join(list(sa & sb)[:5])})

    agree = pd.DataFrame(rows)
    print("EXPLANATION AGREEMENT — SHAP TOP-10 OVERLAP")
    print("="*80)
    print(agree.to_string(index=False))

    mean_jac = agree["Jaccard"].mean()
    print(f"\nMean Jaccard: {mean_jac:.3f}")
    if mean_jac > 0.60:
        print("  High agreement — the model uses the same features everywhere,")
        print("  so subgroup failure is about how those features behave,")
        print("  not about which features are used.")
    elif mean_jac < 0.30:
        print("  Low agreement — different subgroups trigger genuinely different")
        print("  features. Findings should be stated per subgroup.")
    else:
        print("  Moderate agreement.")

    expl_score = score_explainability_risk(3 if mean_jac > 0.60 else
                                           (2 if mean_jac > 0.30 else 1))
    print(f"\nExplainability Risk score: {expl_score} / 3")
    save_result_table(agree, "Table12b_Explanation_Agreement")
except FileNotFoundError:
    print("SHAP profiles not found. Run Notebook 9 Cell 6 first.")
    expl_score = 2

EXPLANATION AGREEMENT — SHAP TOP-10 OVERLAP
 Subgroup A  Subgroup B  Jaccard                              Shared Features
     formal emoji-heavy    0.250            punct_intensity, it, to, char_ i 
     formal slang-heavy    0.111                    punct_intensity, char_ i 
     formal     sarcasm    0.111                         punct_intensity, not
emoji-heavy slang-heavy    0.250 punct_intensity, love, slang_ratio, char_ i 
emoji-heavy     sarcasm    0.111                       punct_intensity, great
slang-heavy     sarcasm    0.053                              punct_intensity

Mean Jaccard: 0.148
  Low agreement — different subgroups trigger genuinely different
  features. Findings should be stated per subgroup.

Explainability Risk score: 3 / 3
  saved table -> Table12b_Explanation_Agreement.csv


## Cell 8: Table 12

In [11]:
total = len(sample)
rows = []
for (ttype, tname), grp in sample.groupby(["taxonomy_type", "taxonomy_name"]):
    main_sg = grp["subgroup"].value_counts().index[0]
    example = grp.nlargest(1, "confidence").iloc[0]
    rules = grp[grp["anchor_rule"] != ""]["anchor_rule"]
    rows.append({
        "Taxonomy Type":            ttype,
        "Name":                     tname,
        "Subgroup Mainly Affected": main_sg,
        "N Cases":                  len(grp),
        "% of Total Errors":        round(len(grp) / total * 100, 1),
        "Example Text":             str(example["text"])[:110],
        "Top SHAP Feature":         example["top_shap_feature"],
        "Anchor Rule":              rules.iloc[0] if len(rules) else "",
        "Anchor Precision":         round(grp["anchor_precision"].mean(), 3)
                                     if grp["anchor_precision"].notna().any() else np.nan,
        "Mean Confidence":          round(grp["confidence"].mean(), 3),
    })

table12 = pd.DataFrame(rows).sort_values("N Cases", ascending=False)

print("="*110)
print("TABLE 12 — MISCLASSIFICATION ERROR TAXONOMY")
print("="*110)
print(table12[["Taxonomy Type", "Name", "Subgroup Mainly Affected",
               "N Cases", "% of Total Errors", "Mean Confidence"]].to_string(index=False))

dom = table12.iloc[0]
print(f"\nDominant error type: {dom['Name']} "
      f"({dom['N Cases']} cases, {dom['% of Total Errors']}%)")
if dom["Taxonomy Type"] == "Type 3":
    print("\nSarcasm Inversion dominates. The model cannot invert surface")
    print("sentiment. Unigram TF-IDF has no mechanism for this, so the cause")
    print("is the feature representation — model selection will not fix it.")
elif dom["Taxonomy Type"] == "Type 1":
    print("\nLexical Trap dominates. Individual sentimental words override")
    print("context. Again a TF-IDF property rather than a model property.")

save_result_table(table12, "Table12_Error_Taxonomy")
sample.to_parquet(PATHS["results"] / "error_taxonomy_instances.parquet", index=False)
print("\nNovelty 3 complete.")

TABLE 12 — MISCLASSIFICATION ERROR TAXONOMY
Taxonomy Type                       Name Subgroup Mainly Affected  N Cases  % of Total Errors  Mean Confidence
       Type 1               Lexical Trap                   formal       18               54.5            0.885
       Type 2   Emoji Polarity Confusion              emoji-heavy       10               30.3            0.961
       Type 4 Noise and Slang Distortion              slang-heavy        3                9.1            0.806
       Type 3          Sarcasm Inversion              slang-heavy        2                6.1            0.959

Dominant error type: Lexical Trap (18 cases, 54.5%)

Lexical Trap dominates. Individual sentimental words override
context. Again a TF-IDF property rather than a model property.
  saved table -> Table12_Error_Taxonomy.csv

Novelty 3 complete.
